In [ ]:
!pip -q install opendatasets
import opendatasets as od

od.download("https://www.kaggle.com/datasets/shuhengmo/uber-nyc-forhire-vehicles-trip-data-2021")



Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: nojusedrwfgasdf
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/shuhengmo/uber-nyc-forhire-vehicles-trip-data-2021


100%|██████████| 4.23G/4.23G [00:51<00:00, 87.8MB/s]


In [ ]:
from pyspark.sql import SparkSession
import time

# Sukuriame Spark sesiją
spark = SparkSession.builder.appName("TaxiDataAnalysis").getOrCreate()
spark

In [ ]:
FAILU_KELIAS = "/content/uber-nyc-forhire-vehicles-trip-data-2021/fhvhv_tripdata_2021-*.parquet"

print("Pradedamas 12 mėnesių nuskaitymas ir apjungimas...")
start_time = time.time()

# Nuskaityti ir apjungti visus 12 mėnesių Parquet failus
duomenys = spark.read.parquet(FAILU_KELIAS)

# Sukelti veiksmą (action) – suskaičiuoti įrašus
bendras_irasu_skaicius = duomenys.count()

end_time = time.time()
pyspark_laikas = end_time - start_time

print(f"\nBendras įrašų skaičius (12 mėn.): {bendras_irasu_skaicius}")
print(f"PySpark nuskaitymo laikas: {pyspark_laikas:.3f} sek.")


Pradedamas 12 mėnesių nuskaitymas ir apjungimas...

Bendras įrašų skaičius (12 mėn.): 174596652
PySpark nuskaitymo laikas: 1.854 sek.


In [ ]:
duomenys.printSchema()


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [ ]:
duomenys.agg({"trip_time": "max"}).show()


+--------------+
|max(trip_time)|
+--------------+
|        240764|
+--------------+



In [ ]:
duomenys.filter(duomenys.tips > 100).count()


828

In [ ]:
duomenys.filter((duomenys.trip_miles <= 2) & (duomenys.base_passenger_fare > 50)).count()


23270

In [ ]:
duomenys.filter((duomenys.trip_miles <= 2) & (duomenys.base_passenger_fare > 100)).count()


2873

In [ ]:
duomenys.filter(duomenys.tips > 100).select("trip_miles","base_passenger_fare","tips","pickup_datetime","dropoff_datetime").orderBy(duomenys.tips.desc()).show(10, truncate=False)


+----------+-------------------+------+-------------------+-------------------+
|trip_miles|base_passenger_fare|tips  |pickup_datetime    |dropoff_datetime   |
+----------+-------------------+------+-------------------+-------------------+
|409.26    |1493.98            |1000.0|2021-05-13 20:11:57|2021-05-14 03:27:15|
|4.56      |26.95              |500.0 |2021-04-17 16:06:28|2021-04-17 16:36:34|
|13.78     |37.26              |450.0 |2021-06-07 22:34:50|2021-06-07 23:05:05|
|5.51      |20.15              |434.57|2021-05-19 15:12:08|2021-05-19 15:42:55|
|49.57     |392.7              |400.0 |2021-10-22 11:04:31|2021-10-22 13:35:06|
|5.92      |19.93              |400.0 |2021-05-27 21:01:52|2021-05-27 21:14:00|
|225.63    |1230.15            |318.79|2021-04-08 17:53:07|2021-04-08 22:09:43|
|24.04     |349.76             |300.0 |2021-09-01 23:34:00|2021-09-02 01:59:40|
|25.21     |69.3               |300.0 |2021-03-12 15:37:10|2021-03-12 17:22:49|
|103.39    |1013.47            |285.28|2

In [ ]:
duomenys.filter((duomenys.trip_miles <= 2) & (duomenys.base_passenger_fare > 50)) \
  .select("trip_miles","base_passenger_fare","trip_time","pickup_datetime","dropoff_datetime") \
  .orderBy(duomenys.base_passenger_fare.desc()).show(10, truncate=False)


+----------+-------------------+---------+-------------------+-------------------+
|trip_miles|base_passenger_fare|trip_time|pickup_datetime    |dropoff_datetime   |
+----------+-------------------+---------+-------------------+-------------------+
|0.306     |1453.5             |49081    |2021-10-30 12:40:04|2021-10-30 12:40:16|
|0.01      |1020.72            |690      |2021-01-22 07:00:45|2021-01-22 07:12:15|
|0.0       |764.74             |13785    |2021-04-29 12:34:55|2021-04-29 16:24:40|
|0.0       |600.86             |13389    |2021-05-28 11:00:01|2021-05-28 14:43:10|
|0.0       |597.65             |9602     |2021-05-21 10:39:24|2021-05-21 13:19:26|
|0.0       |573.39             |9126     |2021-05-22 11:19:04|2021-05-22 13:51:10|
|0.0       |559.21             |424      |2021-02-28 20:53:47|2021-02-28 21:00:51|
|0.0       |545.64             |14328    |2021-05-07 14:05:14|2021-05-07 18:04:02|
|0.0       |517.61             |13602    |2021-05-07 15:29:11|2021-05-07 19:15:53|
|0.0

In [ ]:
duomenys.filter((duomenys.trip_miles <= 2) & (duomenys.base_passenger_fare > 100)) \
  .select("trip_miles","base_passenger_fare","trip_time","pickup_datetime","dropoff_datetime") \
  .orderBy(duomenys.base_passenger_fare.desc()).show(10, truncate=False)


+----------+-------------------+---------+-------------------+-------------------+
|trip_miles|base_passenger_fare|trip_time|pickup_datetime    |dropoff_datetime   |
+----------+-------------------+---------+-------------------+-------------------+
|0.306     |1453.5             |49081    |2021-10-30 12:40:04|2021-10-30 12:40:16|
|0.01      |1020.72            |690      |2021-01-22 07:00:45|2021-01-22 07:12:15|
|0.0       |764.74             |13785    |2021-04-29 12:34:55|2021-04-29 16:24:40|
|0.0       |600.86             |13389    |2021-05-28 11:00:01|2021-05-28 14:43:10|
|0.0       |597.65             |9602     |2021-05-21 10:39:24|2021-05-21 13:19:26|
|0.0       |573.39             |9126     |2021-05-22 11:19:04|2021-05-22 13:51:10|
|0.0       |559.21             |424      |2021-02-28 20:53:47|2021-02-28 21:00:51|
|0.0       |545.64             |14328    |2021-05-07 14:05:14|2021-05-07 18:04:02|
|0.0       |517.61             |13602    |2021-05-07 15:29:11|2021-05-07 19:15:53|
|0.0